# Retail Store DNA Builder - Stage 5 Output Reader



**Reads from:** `data/USA_100_Stores/store_dna/`


## 1. Setup

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

STORE_DNA_DIR = PROJECT_ROOT / "data" / "USA_100_Stores" / "store_dna"

print("Project root:", PROJECT_ROOT)
print("Stage 5 output directory:", STORE_DNA_DIR)


Project root: d:\AICOE\Retail-StoreDNA
Stage 5 output directory: d:\AICOE\Retail-StoreDNA\data\USA_100_Stores\store_dna


## 2. Read the Stage 5 manifest

In [2]:
manifest = json.loads((STORE_DNA_DIR / "store_dna_manifest.json").read_text(encoding="utf-8"))
print(json.dumps(manifest, indent=2))


{
  "fusion_method": "weighted_late_fusion_concat",
  "l2_normalize_inputs": true,
  "l2_normalize_output": true,
  "weights": {
    "reviews": 1.0,
    "news": 1.0,
    "reports": 1.0,
    "products": 1.0,
    "ops_weekly": 1.0,
    "structured_ops": 0.5
  },
  "store_count": 100,
  "store_dna_dim": 15365,
  "modalities": [
    {
      "modality": "reviews",
      "vector_dim": 3072,
      "weight": 1.0
    },
    {
      "modality": "news",
      "vector_dim": 3072,
      "weight": 1.0
    },
    {
      "modality": "reports",
      "vector_dim": 3072,
      "weight": 1.0
    },
    {
      "modality": "products",
      "vector_dim": 3072,
      "weight": 1.0
    },
    {
      "modality": "ops_weekly",
      "vector_dim": 3072,
      "weight": 1.0
    },
    {
      "modality": "structured_ops",
      "vector_dim": 5,
      "weight": 0.5
    }
  ],
  "outputs": {
    "vectors": "store_dna_vectors.npz",
    "index": "store_dna_index.csv"
  },
  "elapsed_sec": 0.9
}


## 3. Read the store-level QC/index file

In [3]:
index_df = pd.read_csv(STORE_DNA_DIR / "store_dna_index.csv")
print("Rows:", len(index_df))
index_df.head(10)


Rows: 100


,store_id,store_dna_dim,store_dna_norm,reviews_present,reviews_norm,news_present,news_norm,reports_present,reports_norm,products_present,products_norm,ops_weekly_present,ops_weekly_norm,structured_ops_present,structured_ops_norm
0,USR-001,15365,1.0,True,1.0,True,1.0,True,1.0,True,1.0,True,1.0,True,3.179605
1,USR-002,15365,1.0,True,1.0,True,1.0,True,1.0,True,1.0,True,1.0,True,2.408151
2,USR-003,15365,1.0,True,1.0,True,1.0,True,1.0,True,1.0,True,1.0,True,1.860113
3,USR-004,15365,1.0,True,1.0,True,1.0,True,1.0,True,1.0,True,1.0,True,3.073683
4,USR-005,15365,1.0,True,1.0,True,1.0,True,1.0,True,1.0,True,1.0,True,1.877908
5,USR-006,15365,1.0,True,1.0,True,1.0,True,1.0,True,1.0,True,1.0,True,1.535270
6,USR-007,15365,1.0,True,1.0,True,1.0,True,1.0,True,1.0,True,1.0,True,1.473108
7,USR-008,15365,1.0,True,1.0,True,1.0,True,1.0,True,1.0,True,1.0,True,2.573668
8,USR-009,15365,1.0,True,1.0,True,1.0,True,1.0,True,1.0,True,1.0,True,2.429336
9,USR-010,15365,1.0,True,1.0,True,1.0,True,1.0,True,1.0,True,1.0,True,2.072793


In [4]:
index_df.describe(include="all").transpose()


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
store_id,100,100,USR-001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
store_dna_dim,100.0,NaN,NaN,NaN,15365.0,0.0,15365.0,15365.0,15365.0,15365.0,15365.0
store_dna_norm,100.0,NaN,NaN,NaN,1.0,0.0,1.0,1.0,1.0,1.0,1.0
reviews_present,100,1,True,100,NaN,NaN,NaN,NaN,NaN,NaN,NaN
reviews_norm,100.0,NaN,NaN,NaN,1.0,0.0,1.0,1.0,1.0,1.0,1.0
news_present,100,1,True,100,NaN,NaN,NaN,NaN,NaN,NaN,NaN
news_norm,100.0,NaN,NaN,NaN,1.0,0.0,1.0,1.0,1.0,1.0,1.0
reports_present,100,1,True,100,NaN,NaN,NaN,NaN,NaN,NaN,NaN
reports_norm,100.0,NaN,NaN,NaN,1.0,0.0,1.0,1.0,1.0,1.0,1.0
products_present,100,1,True,100,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 4. Read the fused StoreDNA vectors

In [5]:
store_dna_npz = np.load(STORE_DNA_DIR / "store_dna_vectors.npz")
store_ids = store_dna_npz["store_ids"]
vectors = store_dna_npz["vectors"]

print("store_ids shape:", store_ids.shape)
print("vectors shape:", vectors.shape)
print("dtype:", vectors.dtype)
print("Average L2 norm:", float(np.linalg.norm(vectors, axis=1).mean()))


store_ids shape: (100,)
vectors shape: (100, 15365)
dtype: float32
Average L2 norm: 1.0


In [6]:
preview_df = pd.DataFrame({
    "store_id": store_ids[:10],
    "first_5_dims": [row[:5].tolist() for row in vectors[:10]],
})
preview_df


,store_id,first_5_dims
0,USR-001,"[-0.0074663832783699036, -0.002070378744974732..."
1,USR-002,"[-0.003789125708863139, -0.00127704709302634, ..."
2,USR-003,"[-0.0035339437890797853, -0.005748020485043526..."
3,USR-004,"[-0.004731175489723682, 0.004823190160095692, ..."
4,USR-005,"[-0.0068034762516617775, 0.008411678485572338,..."
5,USR-006,"[-0.005905381869524717, 0.0020792719442397356,..."
6,USR-007,"[-0.004078138154000044, 0.0017959600081667304,..."
7,USR-008,"[-0.004134508781135082, 0.007493413984775543, ..."
8,USR-009,"[-0.008660530671477318, 0.005797314457595348, ..."
9,USR-010,"[0.002156407805159688, 0.004740290809422731, -..."


## 5. Read one store by ID

In [8]:
TARGET_STORE_ID = "USR-001"

match_idx = np.where(store_ids == TARGET_STORE_ID)[0]
if len(match_idx) == 0:
    print(f"Store {TARGET_STORE_ID} not found")
else:
    i = int(match_idx[0])
    print("Store:", store_ids[i])
    print("Vector length:", len(vectors[i]))
    print("First 20 dims:", vectors[i][:20])
    display(index_df[index_df["store_id"] == TARGET_STORE_ID])


Store: USR-001
Vector length: 15365
First 20 dims: [-0.00746638 -0.00207038 -0.00141402  0.00878851  0.0065873  -0.02094626
  0.00900953 -0.00213888  0.00987682  0.01524053  0.01132631 -0.00855855
 -0.0079307  -0.0186773  -0.00765223  0.0084405  -0.01308012 -0.00282714
  0.00130756  0.00739908]


,store_id,store_dna_dim,store_dna_norm,reviews_present,reviews_norm,news_present,news_norm,reports_present,reports_norm,products_present,products_norm,ops_weekly_present,ops_weekly_norm,structured_ops_present,structured_ops_norm
0,USR-001,15365,1.0,True,1.0,True,1.0,True,1.0,True,1.0,True,1.0,True,3.179605


## 6. Simple similarity example

In [9]:
TARGET_STORE_ID = "USR-001"

match_idx = np.where(store_ids == TARGET_STORE_ID)[0]
if len(match_idx) == 0:
    print(f"Store {TARGET_STORE_ID} not found")
else:
    i = int(match_idx[0])
    query = vectors[i]
    scores = vectors @ query
    top_idx = np.argsort(scores)[::-1][:10]
    similar_df = pd.DataFrame({
        "store_id": store_ids[top_idx],
        "similarity": scores[top_idx],
    })
    similar_df


## 7. Output files

```
data/USA_100_Stores/store_dna/
├── store_dna_vectors.npz
├── store_dna_index.csv
└── store_dna_manifest.json
```
